# LeetCode #31 — Next Permutation: tìm hoán vị kế tiếp theo thứ tự từ điển

**Mức độ:** Medium · **Chủ đề:** Array, Two Pointers

Skeleton trên LeetCode:

```python
class Solution:
    def nextPermutation(self, nums: List[int]) -> None:
        """
        Do not return anything, modify nums in-place instead.
        """
```

Bài này có một thuật toán ngắn (khoảng 10 dòng) nhưng **khó tự nghĩ ra** nếu không hiểu bản chất của **thứ tự từ điển**. Mục tiêu của notebook là giúp bạn **suy ra** thuật toán, chứ không học thuộc 3 bước.

---

## Sau notebook này, bạn sẽ nắm được

1. Thứ tự từ điển (lexicographic order) trên dãy số.
2. Quan sát then chốt: **hậu tố giảm dần** là "lớn nhất có thể" và không thể tăng thêm.
3. Thuật toán 3 bước: **tìm pivot → đổi chỗ với phần tử lớn hơn nhỏ nhất bên phải → đảo hậu tố**.
4. Cách sửa mảng **tại chỗ (in-place)** với bộ nhớ `O(1)`.

# 1. Bài toán là gì?

Một **hoán vị** của mảng là một cách sắp xếp lại các phần tử. Nếu liệt kê mọi hoán vị theo **thứ tự từ điển**, **hoán vị kế tiếp** là hoán vị đứng ngay sau.

Nếu mảng đã là hoán vị **lớn nhất**, hoán vị kế tiếp được quy ước là hoán vị **nhỏ nhất** (sắp xếp tăng dần).

Phải sửa `nums` **tại chỗ**, chỉ dùng bộ nhớ phụ hằng số.

**Constraints (theo đề):**

- `1 <= nums.length <= 100`
- `0 <= nums[i] <= 100`

---

## Ví dụ

```text
[1, 2, 3]  ->  [1, 3, 2]
[3, 2, 1]  ->  [1, 2, 3]     (lớn nhất -> quay về nhỏ nhất)
[1, 1, 5]  ->  [1, 5, 1]
```

Các hoán vị của `[1, 2, 3]` theo thứ tự từ điển:

```text
[1,2,3] -> [1,3,2] -> [2,1,3] -> [2,3,1] -> [3,1,2] -> [3,2,1] -> (quay lại) [1,2,3]
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Hoán vị, tổ hợp, tập con và thứ tự từ điển
2. Hai con trỏ (two pointers)

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Hoán vị, tổ hợp, tập con và thứ tự từ điển

### Đếm

| Đối tượng | Ý nghĩa | Số lượng |
|---|---|---|
| Hoán vị của `n` phần tử | sắp **thứ tự** tất cả | `n!` |
| Chỉnh hợp chập `k` | chọn `k` phần tử **có thứ tự** | `n! / (n−k)!` |
| Tổ hợp chập `k` | chọn `k` phần tử **không** thứ tự | `C(n, k) = n! / (k! (n−k)!)` |
| Tập con | mỗi phần tử lấy/không | `2^n` |
| Hoán vị có lặp | `n` phần tử, nhóm trùng cỡ `c1, c2, …` | `n! / (c1! c2! …)` |

In [1]:
from math import factorial, comb, perm
from itertools import permutations, combinations, product

print("4! =", factorial(4), "| P(4,2) =", perm(4, 2), "| C(4,2) =", comb(4, 2), "| 2^4 =", 2 ** 4)
print("hoán vị [1,2,3]:", list(permutations([1, 2, 3])))
print("tổ hợp chập 2 của [1,2,3,4]:", list(combinations([1, 2, 3, 4], 2)))
print("hoán vị khác nhau của 'aab':", sorted(set(permutations("aab"))), "=", factorial(3) // factorial(2))
print("tích Descartes {0,1}^2:", list(product([0, 1], repeat=2)))

4! = 24 | P(4,2) = 12 | C(4,2) = 6 | 2^4 = 16
hoán vị [1,2,3]: [(1, 2, 3), (1, 3, 2), (2, 1, 3), (2, 3, 1), (3, 1, 2), (3, 2, 1)]
tổ hợp chập 2 của [1,2,3,4]: [(1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)]
hoán vị khác nhau của 'aab': [('a', 'a', 'b'), ('a', 'b', 'a'), ('b', 'a', 'a')] = 3
tích Descartes {0,1}^2: [(0, 0), (0, 1), (1, 0), (1, 1)]


### Thứ tự từ điển (lexicographic order)

So hai dãy như so từ trong từ điển: so phần tử **đầu tiên khác nhau**; nếu một dãy là tiền tố của dãy kia, dãy ngắn hơn đứng trước. Python so sánh list/tuple/str đúng theo kiểu này.

Liệt kê hoán vị của `[1, 2, 3]` theo thứ tự từ điển: `123, 132, 213, 231, 312, 321`.

**Quan sát về hoán vị kế tiếp:** đi từ phải sang, phần **hậu tố giảm dần** dài nhất là hậu tố "đã lớn nhất có thể". Muốn hoán vị lớn hơn kế tiếp, phải tăng phần tử **ngay trước** hậu tố đó lên một chút.

In [2]:
print([1, 2, 3] < [1, 3], "|", "abc" < "abd", "|", [1, 2] < [1, 2, 0], "|", "10" < "9")
for p in permutations([1, 2, 3]):
    i = len(p) - 1
    while i > 0 and p[i - 1] >= p[i]:
        i -= 1
    print(p, "| hậu tố giảm dần dài nhất:", p[i:])

True | True | True | True
(1, 2, 3) | hậu tố giảm dần dài nhất: (3,)
(1, 3, 2) | hậu tố giảm dần dài nhất: (3, 2)
(2, 1, 3) | hậu tố giảm dần dài nhất: (3,)
(2, 3, 1) | hậu tố giảm dần dài nhất: (3, 1)
(3, 1, 2) | hậu tố giảm dần dài nhất: (2,)
(3, 2, 1) | hậu tố giảm dần dài nhất: (3, 2, 1)


### Liên hệ với quay lui

`itertools` sinh sẵn hoán vị/tổ hợp, nhưng phỏng vấn thường yêu cầu **tự sinh** bằng quay lui (để kèm cắt tỉa, bỏ trùng, điều kiện riêng). Kích thước kết quả (`n!`, `C(n,k)`, `2^n`) cho biết độ phức tạp tối thiểu — không thuật toán nào liệt kê nhanh hơn số lời giải.

## 2.2 Hai con trỏ (two pointers)

### Ý tưởng

Dùng **hai chỉ số** di chuyển trên mảng/chuỗi theo quy tắc, để thay vòng lặp lồng `O(n²)` bằng một lượt `O(n)`. Hai kiểu chính:

**1. Hai đầu tiến vào nhau** (thường trên mảng đã sắp xếp):

```text
[1, 2, 4, 7, 11, 15], tìm hai số có tổng 15
 L                 R     1 + 15 = 16 > 15 → R lùi (cần nhỏ hơn)
 L             R         1 + 11 = 12 < 15 → L tiến (cần lớn hơn)
    L          R         2 + 11 = 13 < 15 → L tiến
       L       R         4 + 11 = 15 ✓
```

Vì sao được phép bỏ? Khi `a[L] + a[R] > target`, mọi cặp `(L', R)` với `L' > L` còn lớn hơn → `R` không thể là đáp án, bỏ `R`.

**2. Cùng chiều — con trỏ đọc và con trỏ ghi** (sửa mảng tại chỗ):

```text
xóa mọi số 0, giữ thứ tự: [0, 1, 0, 3, 12]
read đi qua từng phần tử; gặp số khác 0 thì chép vào vị trí write, write += 1
→ [1, 3, 12, ...]
```

In [3]:
def tong_hai_so_da_sap_xep(a, target):
    L, R = 0, len(a) - 1
    while L < R:
        s = a[L] + a[R]
        print(f"   L={L} R={R}: {a[L]} + {a[R]} = {s}")
        if s == target:
            return L, R
        if s < target:
            L += 1
        else:
            R -= 1
    return None

print("→", tong_hai_so_da_sap_xep([1, 2, 4, 7, 11, 15], 15))

def bo_so_khong(a):
    write = 0
    for read in range(len(a)):
        if a[read] != 0:
            a[write] = a[read]
            write += 1
    return a[:write]

print("bỏ số 0:", bo_so_khong([0, 1, 0, 3, 12]))

def la_palindrome(s):
    i, j = 0, len(s) - 1
    while i < j:
        if s[i] != s[j]:
            return False
        i += 1; j -= 1
    return True
print("'racecar' palindrome?", la_palindrome("racecar"), "| 'abca'?", la_palindrome("abca"))

   L=0 R=5: 1 + 15 = 16
   L=0 R=4: 1 + 11 = 12
   L=1 R=4: 2 + 11 = 13
   L=2 R=4: 4 + 11 = 15
→ (2, 4)
bỏ số 0: [1, 3, 12]
'racecar' palindrome? True | 'abca'? False


### Đảo ngược một đoạn tại chỗ

Hai con trỏ ở hai đầu, đổi chỗ rồi tiến vào trong — dùng rất nhiều (đảo mảng, xoay mảng, đảo từ).

In [4]:
def dao_doan(a, i, j):
    while i < j:
        a[i], a[j] = a[j], a[i]
        i += 1; j -= 1

arr_tp = [1, 2, 3, 4, 5, 6]
dao_doan(arr_tp, 1, 4)
print("đảo đoạn [1..4]:", arr_tp)

đảo đoạn [1..4]: [1, 5, 4, 3, 2, 6]


### Lỗi hay gặp

- Quên **sắp xếp** trước khi dùng hai đầu cho bài tổng.
- Điều kiện `L <= R` thay vì `L < R` → dùng một phần tử hai lần.
- Không chứng minh được vì sao bỏ một đầu là an toàn → dễ bỏ mất đáp án.
- Con trỏ ghi vượt con trỏ đọc (không thể xảy ra nếu `write` chỉ tăng khi ghi).

## 2.3 Áp dụng vào bài này

**Hoán vị kế tiếp theo thứ tự từ điển**: tìm từ phải sang phần **hậu tố giảm dần** dài nhất (đã "lớn nhất"), phần tử ngay trước nó là pivot; đổi pivot với phần tử nhỏ nhất lớn hơn nó trong hậu tố, rồi **đảo** hậu tố (đảo đoạn bằng **hai con trỏ**) để nó nhỏ nhất.

# 3. Nền tảng: thứ tự từ điển

So sánh hai dãy giống so sánh từ trong từ điển: tìm **vị trí đầu tiên khác nhau**, dãy nào có phần tử nhỏ hơn ở đó thì nhỏ hơn.

```text
[1, 3, 2]  vs  [2, 1, 3]   -> khác ở vị trí 0: 1 < 2 -> [1,3,2] nhỏ hơn
[2, 1, 3]  vs  [2, 3, 1]   -> khác ở vị trí 1: 1 < 3 -> [2,1,3] nhỏ hơn
```

Python so sánh list đúng theo cách này.

In [5]:
print([1, 3, 2] < [2, 1, 3])
print([2, 1, 3] < [2, 3, 1])
print(sorted([[2, 3, 1], [1, 2, 3], [3, 1, 2], [1, 3, 2]]))

True
True
[[1, 2, 3], [1, 3, 2], [2, 3, 1], [3, 1, 2]]


**Hệ quả quan trọng:** thay đổi ở vị trí **càng bên phải** thì thay đổi giá trị **càng ít** (giống số `1239` → `1293` tăng ít hơn `1239` → `2139`).

→ Để có hoán vị **kế tiếp** (lớn hơn **ít nhất**), ta muốn thay đổi **càng xa về bên phải càng tốt**.

# 4. Cách dễ nghĩ nhất: liệt kê mọi hoán vị

Dùng `itertools.permutations`, sắp xếp, tìm vị trí của `nums` rồi lấy phần tử kế tiếp.

In [6]:
from typing import List
from itertools import permutations

def next_permutation_bruteforce(nums: List[int]) -> List[int]:
    all_perms = sorted(set(permutations(nums)))     # set để bỏ hoán vị trùng khi có giá trị trùng
    idx = all_perms.index(tuple(nums))
    return list(all_perms[(idx + 1) % len(all_perms)])

print(next_permutation_bruteforce([1, 2, 3]))
print(next_permutation_bruteforce([3, 2, 1]))
print(next_permutation_bruteforce([1, 1, 5]))

[1, 3, 2]
[1, 2, 3]
[1, 5, 1]


**Vấn đề:** có tới `n!` hoán vị. Với `n = 100`, `100!` có 158 chữ số — không thể liệt kê. Đề còn yêu cầu bộ nhớ `O(1)`.

# 5. Quan sát then chốt: hậu tố giảm dần

Xét `nums = [1, 5, 8, 4, 7, 6, 5, 3, 1]`.

Đọc từ **phải sang trái**: `1, 3, 5, 6, 7` tăng dần… tới `4` thì **giảm**. Tức hậu tố `[7, 6, 5, 3, 1]` là **giảm dần** (không tăng).

**Một dãy giảm dần là hoán vị lớn nhất của chính nó.** Không có cách nào sắp xếp lại `[7, 6, 5, 3, 1]` để được dãy lớn hơn.

→ Muốn lớn hơn, **bắt buộc** phải thay đổi phần tử **ngay trước** hậu tố đó — ở đây là `4` (vị trí 3). Gọi nó là **pivot**.

```text
[1, 5, 8, 4, | 7, 6, 5, 3, 1]
          ^      hậu tố giảm dần
        pivot
```

In [7]:
nums = [1, 5, 8, 4, 7, 6, 5, 3, 1]
i = len(nums) - 2
while i >= 0 and nums[i] >= nums[i + 1]:
    i -= 1
print("pivot ở chỉ số", i, "giá trị", nums[i])
print("hậu tố:", nums[i + 1:])

pivot ở chỉ số 3 giá trị 4
hậu tố: [7, 6, 5, 3, 1]


# 6. Thay pivot bằng gì?

Pivot `4` phải được thay bằng một số **lớn hơn 4** lấy từ hậu tố — và để tăng **ít nhất**, chọn số **nhỏ nhất** trong các số lớn hơn 4. Trong `[7, 6, 5, 3, 1]`, đó là `5`.

Vì hậu tố giảm dần, ta tìm từ **phải sang trái**: phần tử đầu tiên `> pivot` chính là số nhỏ nhất lớn hơn pivot.

**Đổi chỗ** `4` và `5`:

```text
[1, 5, 8, 5, | 7, 6, 4, 3, 1]
```

Hậu tố **vẫn giảm dần** (vì `4` chen vào đúng chỗ của `5`, giữa `6` và `3`).

# 7. Làm hậu tố nhỏ nhất

Bây giờ phần đầu `[1, 5, 8, 5]` đã lớn hơn bản gốc. Để hoán vị lớn hơn **ít nhất**, phần hậu tố phải **nhỏ nhất có thể** — tức sắp xếp **tăng dần**.

Hậu tố đang giảm dần, nên chỉ cần **đảo ngược** nó (`O(n)`, không cần sắp xếp):

```text
[1, 5, 8, 5, | 1, 3, 4, 6, 7]
```

Đó là đáp án.

## Trường hợp không có pivot

Nếu cả mảng giảm dần (như `[3, 2, 1]`), không có pivot (`i = -1`): đây là hoán vị lớn nhất. Đảo ngược cả mảng → nhỏ nhất. Đúng quy ước của đề!

# 8. Thuật toán 3 bước

```text
1. Tìm pivot i: từ phải sang trái, vị trí đầu tiên có nums[i] < nums[i+1].
2. Nếu có pivot: từ phải sang trái tìm j đầu tiên có nums[j] > nums[i]; đổi chỗ nums[i], nums[j].
3. Đảo ngược đoạn nums[i+1 :] (nếu không có pivot, i = -1, đảo cả mảng).
```

In [8]:
def next_permutation(nums: List[int]) -> None:
    n = len(nums)

    # Bước 1: tìm pivot
    i = n - 2
    while i >= 0 and nums[i] >= nums[i + 1]:
        i -= 1

    # Bước 2: đổi pivot với số nhỏ nhất lớn hơn nó trong hậu tố
    if i >= 0:
        j = n - 1
        while nums[j] <= nums[i]:
            j -= 1
        nums[i], nums[j] = nums[j], nums[i]

    # Bước 3: đảo ngược hậu tố
    left, right = i + 1, n - 1
    while left < right:
        nums[left], nums[right] = nums[right], nums[left]
        left += 1
        right -= 1


for arr in [[1, 2, 3], [3, 2, 1], [1, 1, 5], [1, 5, 8, 4, 7, 6, 5, 3, 1]]:
    original = arr[:]
    next_permutation(arr)          # sửa tại chỗ, không trả về gì
    print(original, "->", arr)

[1, 2, 3] -> [1, 3, 2]
[3, 2, 1] -> [1, 2, 3]
[1, 1, 5] -> [1, 5, 1]
[1, 5, 8, 4, 7, 6, 5, 3, 1] -> [1, 5, 8, 5, 1, 3, 4, 6, 7]


Hàm **không trả về** gì (`None`) — nó sửa trực tiếp list được truyền vào. Đây là ý nghĩa của "in-place".

Bước 3 dùng **hai con trỏ** đổi chỗ từ hai đầu vào giữa. Có thể viết gọn `nums[i + 1:] = reversed(nums[i + 1:])`, nhưng cách đó tạo list tạm (bộ nhớ `O(n)`).

# 9. Mô phỏng từng bước

In [9]:
def trace_next_permutation(nums: List[int]) -> None:
    n = len(nums)
    print("Ban đầu:", nums)
    i = n - 2
    while i >= 0 and nums[i] >= nums[i + 1]:
        i -= 1
    if i < 0:
        print("Bước 1: không có pivot -> mảng giảm dần (hoán vị lớn nhất)")
    else:
        print(f"Bước 1: pivot i = {i} (giá trị {nums[i]}), hậu tố giảm dần = {nums[i + 1:]}")
        j = n - 1
        while nums[j] <= nums[i]:
            j -= 1
        print(f"Bước 2: j = {j} (giá trị {nums[j]}) là số nhỏ nhất > {nums[i]} trong hậu tố -> đổi chỗ")
        nums[i], nums[j] = nums[j], nums[i]
        print(f"        {nums}")
    left, right = i + 1, n - 1
    while left < right:
        nums[left], nums[right] = nums[right], nums[left]
        left += 1
        right -= 1
    print(f"Bước 3: đảo ngược nums[{i + 1}:] -> {nums}")

trace_next_permutation([1, 5, 8, 4, 7, 6, 5, 3, 1])
print()
trace_next_permutation([3, 2, 1])
print()
trace_next_permutation([1, 3, 2])

Ban đầu: [1, 5, 8, 4, 7, 6, 5, 3, 1]
Bước 1: pivot i = 3 (giá trị 4), hậu tố giảm dần = [7, 6, 5, 3, 1]
Bước 2: j = 6 (giá trị 5) là số nhỏ nhất > 4 trong hậu tố -> đổi chỗ
        [1, 5, 8, 5, 7, 6, 4, 3, 1]
Bước 3: đảo ngược nums[4:] -> [1, 5, 8, 5, 1, 3, 4, 6, 7]

Ban đầu: [3, 2, 1]
Bước 1: không có pivot -> mảng giảm dần (hoán vị lớn nhất)
Bước 3: đảo ngược nums[0:] -> [1, 2, 3]

Ban đầu: [1, 3, 2]
Bước 1: pivot i = 0 (giá trị 1), hậu tố giảm dần = [3, 2]
Bước 2: j = 2 (giá trị 2) là số nhỏ nhất > 1 trong hậu tố -> đổi chỗ
        [2, 3, 1]
Bước 3: đảo ngược nums[1:] -> [2, 1, 3]


# 10. Phiên bản đúng format LeetCode

In [10]:
class Solution:
    def nextPermutation(self, nums: List[int]) -> None:
        """
        Do not return anything, modify nums in-place instead.
        """
        n = len(nums)
        i = n - 2
        while i >= 0 and nums[i] >= nums[i + 1]:
            i -= 1
        if i >= 0:
            j = n - 1
            while nums[j] <= nums[i]:
                j -= 1
            nums[i], nums[j] = nums[j], nums[i]
        left, right = i + 1, n - 1
        while left < right:
            nums[left], nums[right] = nums[right], nums[left]
            left += 1
            right -= 1

In [11]:
def check(name, nums, expected):
    sol = Solution()
    original = nums[:]
    result = sol.nextPermutation(nums)
    assert result is None, "phải sửa tại chỗ, không trả về"
    assert nums == expected, (name, original, nums, expected)
    print(f"{name:<26} PASS  {original} -> {nums}")


check("Ví dụ 1", [1, 2, 3], [1, 3, 2])
check("Ví dụ 2: lớn nhất", [3, 2, 1], [1, 2, 3])
check("Ví dụ 3: có trùng", [1, 1, 5], [1, 5, 1])
check("Một phần tử", [1], [1])
check("Toàn bằng nhau", [2, 2, 2], [2, 2, 2])
check("Trùng ở hậu tố", [1, 5, 1], [5, 1, 1])
check("Pivot ở đầu", [2, 3, 1], [3, 1, 2])
check("Dài", [1, 5, 8, 4, 7, 6, 5, 3, 1], [1, 5, 8, 5, 1, 3, 4, 6, 7])

Ví dụ 1                    PASS  [1, 2, 3] -> [1, 3, 2]
Ví dụ 2: lớn nhất          PASS  [3, 2, 1] -> [1, 2, 3]
Ví dụ 3: có trùng          PASS  [1, 1, 5] -> [1, 5, 1]
Một phần tử                PASS  [1] -> [1]
Toàn bằng nhau             PASS  [2, 2, 2] -> [2, 2, 2]
Trùng ở hậu tố             PASS  [1, 5, 1] -> [5, 1, 1]
Pivot ở đầu                PASS  [2, 3, 1] -> [3, 1, 2]
Dài                        PASS  [1, 5, 8, 4, 7, 6, 5, 3, 1] -> [1, 5, 8, 5, 1, 3, 4, 6, 7]


## Kiểm tra: lặp `nextPermutation` đi qua **mọi** hoán vị đúng thứ tự

Bắt đầu từ mảng đã sắp xếp, gọi liên tục `nextPermutation`; ta phải đi qua đúng các hoán vị phân biệt theo thứ tự tăng dần, rồi quay về đầu.

In [12]:
import random

sol = Solution()
random.seed(31)
for _ in range(300):
    base = sorted(random.randint(0, 3) for _ in range(random.randint(1, 6)))
    expected_order = sorted(set(permutations(base)))
    cur = base[:]
    seen = []
    for _ in range(len(expected_order)):
        seen.append(tuple(cur))
        sol.nextPermutation(cur)
    assert seen == expected_order, base
    assert cur == base                     # quay về hoán vị nhỏ nhất
print("300 mảng ngẫu nhiên (có trùng): đi qua đúng mọi hoán vị theo thứ tự rồi quay về đầu")

300 mảng ngẫu nhiên (có trùng): đi qua đúng mọi hoán vị theo thứ tự rồi quay về đầu


# 11. Vì sao đúng? Độ phức tạp?

Gọi `P` là hoán vị hiện tại, `i` là pivot.

1. **Phần `P[0..i-1]` phải giữ nguyên:** hậu tố `P[i+1..]` đã lớn nhất có thể; mọi hoán vị lớn hơn giữ nguyên `P[0..i]` đều không tồn tại. Vì vậy vị trí thay đổi **sớm nhất** là `i`, và thay đổi ở vị trí muộn nhất có thể cho mức tăng nhỏ nhất.
2. **`P[i]` phải tăng lên ít nhất:** thay bằng số nhỏ nhất lớn hơn nó trong hậu tố.
3. **Hậu tố mới phải nhỏ nhất:** sắp xếp tăng dần — và vì sau khi đổi chỗ hậu tố vẫn giảm dần, đảo ngược là đủ.

Ba lựa chọn "ít nhất" này cho ra đúng hoán vị lớn hơn `P` nhỏ nhất, tức hoán vị kế tiếp.

**Độ phức tạp** (`n = len(nums)`):

```text
Time:  O(n)   — mỗi bước duyệt tối đa n phần tử
Space: O(1)   — chỉ đổi chỗ tại chỗ
```

# 12. Những lỗi dễ mắc

## Lỗi 1: Dùng `>` thay vì `>=` khi tìm pivot (sai với phần tử trùng)

Hậu tố "không tăng" có thể chứa các phần tử **bằng nhau**, nên điều kiện tìm pivot phải bỏ qua cả chúng (`>=`, không phải `>`). Ví dụ `[1, 5, 5]`:

In [13]:
def wrong_pivot_strict(nums: List[int]) -> None:
    n = len(nums)
    i = n - 2
    while i >= 0 and nums[i] > nums[i + 1]:          # SAI: thiếu dấu =
        i -= 1
    if i >= 0:
        j = n - 1
        while nums[j] <= nums[i]:
            j -= 1
        nums[i], nums[j] = nums[j], nums[i]
    nums[i + 1:] = reversed(nums[i + 1:])

a = [1, 5, 5]
try:
    wrong_pivot_strict(a)
    print("Sai :", a)
except IndexError as e:
    print("Sai : IndexError -", e)
b = [1, 5, 5]
next_permutation(b)
print("Đúng:", b)

Sai : IndexError - list index out of range
Đúng: [5, 1, 5]


Với `[1, 5, 5]`, pivot sai rơi vào `i = 1` (vì `5 > 5` là `False`), rồi vòng tìm `j` không tìm được số nào `> 5` → chạy lố ra ngoài mảng.

## Lỗi 2: Sắp xếp hậu tố thay vì đảo ngược

Vẫn **đúng**, nhưng tốn `O(n log n)` và không tận dụng được tính chất "hậu tố giảm dần".

## Lỗi 3: Tìm `j` từ trái sang phải

Trong hậu tố giảm dần, phần tử **đầu tiên** `> pivot` khi đi từ trái là phần tử **lớn nhất**, không phải nhỏ nhất → tăng quá nhiều.

In [14]:
def wrong_j_from_left(nums: List[int]) -> None:
    n = len(nums)
    i = n - 2
    while i >= 0 and nums[i] >= nums[i + 1]:
        i -= 1
    if i >= 0:
        j = i + 1
        while j + 1 < n and nums[j] <= nums[i]:      # SAI: lấy phần tử > pivot đầu tiên từ TRÁI
            j += 1
        nums[i], nums[j] = nums[j], nums[i]
    nums[i + 1:] = reversed(nums[i + 1:])

a = [1, 5, 8, 4, 7, 6, 5, 3, 1]
wrong_j_from_left(a)
print("Sai :", a)
b = [1, 5, 8, 4, 7, 6, 5, 3, 1]
next_permutation(b)
print("Đúng:", b)

Sai : [1, 5, 8, 7, 1, 3, 5, 6, 4]
Đúng: [1, 5, 8, 5, 1, 3, 4, 6, 7]


## Lỗi 4: Trả về list mới thay vì sửa tại chỗ

LeetCode kiểm tra **chính** list `nums` sau khi gọi hàm. Viết `nums = something` bên trong hàm chỉ gán lại biến cục bộ, **không** sửa list gốc. Dùng `nums[:] = ...` hoặc đổi chỗ từng phần tử.

# 13. Mẹo nhận diện pattern

- "Hoán vị kế tiếp / trước đó", "số lớn hơn nhỏ nhất dùng cùng chữ số" (**#556 Next Greater Element III**) → thuật toán 3 bước này.
- **Quét từ phải sang trái** tìm điểm "gãy" của dãy đơn điệu là một kỹ thuật hay dùng khi muốn thay đổi "ít nhất có thể".
- Đảo ngược đoạn tại chỗ bằng hai con trỏ: **#344 Reverse String**, **#189 Rotate Array**.

## Template

```python
i = n - 2
while i >= 0 and nums[i] >= nums[i + 1]: i -= 1
if i >= 0:
    j = n - 1
    while nums[j] <= nums[i]: j -= 1
    nums[i], nums[j] = nums[j], nums[i]
đảo ngược nums[i + 1:]
```

# 14. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Tìm hoán vị kế tiếp của `[2, 4, 3, 1]` và `[1, 3, 5, 4, 2]`, ghi rõ pivot, `j`, mảng sau đổi chỗ và sau đảo ngược.

## Bài 2 — Đảo đoạn tại chỗ

Viết hàm đảo ngược `nums[lo..hi]` (bao gồm hai đầu) tại chỗ bằng hai con trỏ.

```python
a = [1, 2, 3, 4, 5]; reverse_range(a, 1, 3)  ->  a == [1, 4, 3, 2, 5]
```

In [15]:
def exercise_2_reverse_range(nums: List[int], lo: int, hi: int) -> None:
    # TODO
    pass


# a = [1, 2, 3, 4, 5]; exercise_2_reverse_range(a, 1, 3); assert a == [1, 4, 3, 2, 5]
# a = [1, 2]; exercise_2_reverse_range(a, 0, 1); assert a == [2, 1]

## Bài 3 — Điền chỗ trống

In [16]:
def exercise_3_next_permutation(nums: List[int]) -> None:
    n = len(nums)
    # TODO 1: tìm pivot i (nhớ dấu >=)
    # TODO 2: nếu có pivot, tìm j từ phải sang trái, đổi chỗ
    # TODO 3: đảo ngược nums[i+1:]
    pass


# a = [1, 2, 3]; exercise_3_next_permutation(a); assert a == [1, 3, 2]
# a = [3, 2, 1]; exercise_3_next_permutation(a); assert a == [1, 2, 3]
# a = [1, 1, 5]; exercise_3_next_permutation(a); assert a == [1, 5, 1]

## Bài 4 — Biến thể: hoán vị **trước đó**

Viết `prev_permutation(nums)` tại chỗ: hoán vị đứng **ngay trước** theo thứ tự từ điển. Nếu đang nhỏ nhất, chuyển thành lớn nhất.

```text
[1, 3, 2] -> [1, 2, 3]
[1, 2, 3] -> [3, 2, 1]
```

Gợi ý nhanh: đổi chiều mọi phép so sánh.

In [17]:
def exercise_4_prev_permutation(nums: List[int]) -> None:
    # TODO
    pass


# a = [1, 3, 2]; exercise_4_prev_permutation(a); assert a == [1, 2, 3]
# a = [1, 2, 3]; exercise_4_prev_permutation(a); assert a == [3, 2, 1]
# a = [2, 1, 3]; exercise_4_prev_permutation(a); assert a == [1, 3, 2]

## Bài 5 — Mini challenge: Next Greater Element III (LeetCode #556)

Cho số nguyên dương `n`. Tìm số **nhỏ nhất lớn hơn `n`** dùng **đúng các chữ số** của `n`. Nếu không có, hoặc kết quả vượt `2^31 - 1`, trả `-1`.

```python
next_greater(12)         -> 21
next_greater(21)         -> -1
next_greater(230241)     -> 230412
next_greater(2147483486) -> -1      (2147483648 vượt 2^31 - 1)
```

In [18]:
def exercise_5_next_greater(n: int) -> int:
    # TODO
    pass


# assert exercise_5_next_greater(12) == 21
# assert exercise_5_next_greater(21) == -1
# assert exercise_5_next_greater(230241) == 230412
# assert exercise_5_next_greater(2147483486) == -1

# 15. Gợi ý

**Bài 1**
- Mức 1: `[2, 4, 3, 1]`: hậu tố giảm dần là `[4, 3, 1]`, pivot là `2`.
- Mức 2: số nhỏ nhất > 2 trong hậu tố là `3`.

**Bài 3**
- Mức 1: so với code mục 8.

**Bài 4**
- Mức 1: tìm pivot `i` với `nums[i] > nums[i + 1]` (hậu tố **tăng dần** là nhỏ nhất).
- Mức 2: tìm `j` từ phải sang trái đầu tiên có `nums[j] < nums[i]`.
- Mức 3: đổi chỗ rồi đảo ngược hậu tố.

**Bài 5**
- Mức 1: `digits = list(str(n))`, áp dụng next permutation.
- Mức 2: nếu không có pivot → `-1`.
- Mức 3: kiểm tra kết quả `<= 2**31 - 1`.

# 16. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [19]:
trace_next_permutation([2, 4, 3, 1])
print()
trace_next_permutation([1, 3, 5, 4, 2])

Ban đầu: [2, 4, 3, 1]
Bước 1: pivot i = 0 (giá trị 2), hậu tố giảm dần = [4, 3, 1]
Bước 2: j = 2 (giá trị 3) là số nhỏ nhất > 2 trong hậu tố -> đổi chỗ
        [3, 4, 2, 1]
Bước 3: đảo ngược nums[1:] -> [3, 1, 2, 4]

Ban đầu: [1, 3, 5, 4, 2]
Bước 1: pivot i = 1 (giá trị 3), hậu tố giảm dần = [5, 4, 2]
Bước 2: j = 3 (giá trị 4) là số nhỏ nhất > 3 trong hậu tố -> đổi chỗ
        [1, 4, 5, 3, 2]
Bước 3: đảo ngược nums[2:] -> [1, 4, 2, 3, 5]


## Đáp án Bài 2

In [20]:
def answer_2_reverse_range(nums: List[int], lo: int, hi: int) -> None:
    while lo < hi:
        nums[lo], nums[hi] = nums[hi], nums[lo]
        lo += 1
        hi -= 1

a = [1, 2, 3, 4, 5]; answer_2_reverse_range(a, 1, 3); assert a == [1, 4, 3, 2, 5]
a = [1, 2]; answer_2_reverse_range(a, 0, 1); assert a == [2, 1]
print("Bài 2 PASS")

Bài 2 PASS


## Đáp án Bài 3

In [21]:
def answer_3_next_permutation(nums: List[int]) -> None:
    n = len(nums)
    i = n - 2
    while i >= 0 and nums[i] >= nums[i + 1]:
        i -= 1
    if i >= 0:
        j = n - 1
        while nums[j] <= nums[i]:
            j -= 1
        nums[i], nums[j] = nums[j], nums[i]
    answer_2_reverse_range(nums, i + 1, n - 1)

a = [1, 2, 3]; answer_3_next_permutation(a); assert a == [1, 3, 2]
a = [3, 2, 1]; answer_3_next_permutation(a); assert a == [1, 2, 3]
a = [1, 1, 5]; answer_3_next_permutation(a); assert a == [1, 5, 1]
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [22]:
def answer_4_prev_permutation(nums: List[int]) -> None:
    n = len(nums)
    i = n - 2
    while i >= 0 and nums[i] <= nums[i + 1]:
        i -= 1
    if i >= 0:
        j = n - 1
        while nums[j] >= nums[i]:
            j -= 1
        nums[i], nums[j] = nums[j], nums[i]
    answer_2_reverse_range(nums, i + 1, n - 1)

a = [1, 3, 2]; answer_4_prev_permutation(a); assert a == [1, 2, 3]
a = [1, 2, 3]; answer_4_prev_permutation(a); assert a == [3, 2, 1]
a = [2, 1, 3]; answer_4_prev_permutation(a); assert a == [1, 3, 2]
for _ in range(500):
    arr = [random.randint(0, 3) for _ in range(random.randint(1, 6))]
    b = arr[:]
    answer_3_next_permutation(b)
    answer_4_prev_permutation(b)
    assert b == arr                      # prev(next(x)) == x
print("Bài 4 PASS (prev là phép ngược của next trên 500 test)")

Bài 4 PASS (prev là phép ngược của next trên 500 test)

## Đáp án Bài 5

In [23]:
def answer_5_next_greater(n: int) -> int:
    digits = list(str(n))
    i = len(digits) - 2
    while i >= 0 and digits[i] >= digits[i + 1]:
        i -= 1
    if i < 0:
        return -1
    j = len(digits) - 1
    while digits[j] <= digits[i]:
        j -= 1
    digits[i], digits[j] = digits[j], digits[i]
    answer_2_reverse_range(digits, i + 1, len(digits) - 1)
    result = int("".join(digits))
    return result if result <= 2**31 - 1 else -1

assert answer_5_next_greater(12) == 21
assert answer_5_next_greater(21) == -1
assert answer_5_next_greater(230241) == 230412
assert answer_5_next_greater(2147483486) == -1
print("Bài 5 PASS")

Bài 5 PASS


So sánh chuỗi chữ số một ký tự `'0'..'9'` cho cùng kết quả như so sánh số, nên ta dùng thẳng `digits` dạng chuỗi.

# 17. Tổng kết

> Từ phải sang trái tìm **pivot** (chỗ dãy hết giảm dần) → đổi pivot với **số nhỏ nhất lớn hơn nó** ở hậu tố → **đảo ngược** hậu tố. Mỗi bước là một lựa chọn "tăng ít nhất có thể".

---

## Checklist trước khi nộp

- [ ] Tìm pivot với `nums[i] >= nums[i + 1]` (có dấu `=`)?
- [ ] Tìm `j` từ **phải sang trái** với `nums[j] <= nums[i]`?
- [ ] Đảo ngược `nums[i + 1:]` (cả khi `i = -1`)?
- [ ] Sửa **tại chỗ**, không trả về?
- [ ] Đã thử mảng giảm dần, mảng 1 phần tử, có phần tử trùng?